# 神经网络的梯度

In [20]:
import numpy as np

def softmax(x):
    """SoftMax"""
    c = np.max(x)
    exp_x = np.exp(x - c)
    sum_exp_x = np.sum(exp_x)
    return exp_x / sum_exp_x

def cross_entropy_error(y, t):
    """交叉熵误差"""
    delta = 1e-7
    return -np.sum(t * np.log(y + delta))


class SimpleNet:
    def __init__(self) -> None:
        # 使用高斯分布构造一个 2 x 3 的单层网络
        self.W = np.random.randn(2, 3)
        
    def predict(self, x):
        z = np.dot(x, self.W)
        return softmax(z)
    
    def loss(self, x, t):
        y = self.predict(x)
        loss = cross_entropy_error(y, t)
        return loss

net = SimpleNet()
print(net.W)

[[ 0.68885221 -0.91170957 -0.4111408 ]
 [-0.1630305   1.04876441 -0.64499093]]


In [21]:
# 生成一个训练数据
x_train_0 = np.array([0.6, 0.9])
t_train_0 = [0, 0, 1]

In [22]:
y = net.predict(x_train_0)
print(f"预测概率为: {y}")
print(f"预测正确值的概率为: {y[np.argmax(t_train_0)]}")
print(f"损失为: {net.loss(x_train_0, t_train_0)}")


预测概率为: [0.40418737 0.4604282  0.13538443]
预测正确值的概率为: 0.13538443070007924
损失为: 1.9996361739224502


In [23]:
def numerical_gradient(f, x):
    """
    梯度计算
    """
    h = 1e-4 # 0.0001
    grad = np.zeros_like(x)
    
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + h
        fxh1 = f(x) # f(x+h)
        
        x[idx] = tmp_val - h 
        fxh2 = f(x) # f(x-h)
        grad[idx] = (fxh1 - fxh2) / (2*h)
        
        x[idx] = tmp_val # 还原值
        it.iternext()   
        
    return grad

def gradient_decent(f, init_x, lr=0.01, step_num=100):
    x = init_x
    for i in range(step_num):
        grad = numerical_gradient(f, x)
        x -= lr * grad
    return x

In [25]:
# 使用梯度下降法更新权重
def f(w):
    net.W = w
    return net.loss(x_train_0, t_train_0)

newW = gradient_decent(f, net.W, lr=0.01, step_num=1000)
net.W = newW

y = net.predict(x_train_0)
print(f"更新参数后的预测概率为: {y}")
print(f"更新参数后的预测正确值的概率为: {y[np.argmax(t_train_0)]}")
print(f"更新参数后的损失为: {net.loss(x_train_0, t_train_0)}")

更新参数后的预测概率为: [0.03093742 0.03183184 0.93723074]
更新参数后的预测正确值的概率为: 0.9372307402672573
更新参数后的损失为: 0.06482566607465857


In [26]:
net.W

array([[-0.06378801, -1.71654358,  1.14633343],
       [-1.29199083, -0.15848661,  1.69122042]])